In [52]:
import pandas as pd
import numpy as np
user_reviews = pd.read_excel(r'/Users/yg/Documents/Database management/Data Files - database/UserReviewsClean43LIWC.xlsx')

for column in user_reviews.columns:
    if user_reviews[column].nunique() == len(user_reviews): 
        print(column)

In [53]:
pip install openpyxl


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Finding out whether this column will be the primary key - no single column has completely unique, non-missing values. 

In [54]:
summary = pd.DataFrame({
        'Total Reviews': [len(user_reviews)],
        'Unique Users': [user_reviews.nunique()],
        'Missing Values': [user_reviews.isnull().sum().sum()],
        'Duplicate Values': [user_reviews.duplicated().sum()],
})

print(summary)

   Total Reviews                                       Unique Users  \
0         319662  url          11364
idvscore        16
reviewer...   

   Missing Values  Duplicate Values  
0           20793               922  


erd = identification of how our tables are look 

we need to find: columns in the datasets, find the important columns that will be needed in those respective data sets. e.g. table from which director got the most sales - what do we need? Director? Sales? Sum of sales to each director -> How many sales has he/she done in total? (example) 

In [55]:
column_names = user_reviews.columns.tolist()

In [56]:
#printing the top 10 rows of the dataframe
display(user_reviews.head(10))

,url,idvscore,reviewer,dateP,Rev,thumbsUp,thumbsTot,WC,Analytic,Clout,...,Comma,Colon,SemiC,QMark,Exclam,Dash,Quote,Apostro,Parenth,OtherP
0,https://www.metacritic.com/movie/bronson,8,'Longbottom94',"'Apr 25, 2013'",'Many have dismissed this film for not explor...,2,2,113,94.05,32.91,...,4.42,0.0,0.00,0.00,0.00,0.88,0,1.77,0.00,0.00
1,https://www.metacritic.com/movie/bronson,9,'MartinB',"'Oct 13, 2009'",'Anyone who doesn t like this movie simply ju...,0,1,103,57.11,38.53,...,0.97,0.0,0.00,0.00,0.00,0.00,0,1.94,5.83,0.00
2,https://www.metacritic.com/movie/bronson,10,'Jaakko',"'Jul 19, 2012'",'Not sure what to think at this film at first...,1,1,143,75.73,66.27,...,0.70,0.0,0.00,0.00,0.00,0.00,0,1.40,0.00,1.40
3,https://www.metacritic.com/movie/bronson,6,'CapoR',"'Oct 13, 2009'",'Nicely portrayed but it lacks the elements t...,0,1,14,83.81,23.75,...,0.00,0.0,0.00,0.00,0.00,0.00,0,14.29,0.00,0.00
4,https://www.metacritic.com/movie/bronson,8,'OrwellB.',"'Oct 10, 2009'",'Bronson is more than entertainment. It is ar...,0,0,20,93.26,69.14,...,0.00,0.0,0.00,0.00,0.00,0.00,0,10.00,0.00,0.00
5,https://www.metacritic.com/movie/bronson,8,'RONGIU',"'Jun 10, 2011'","'Sorry, machine translation. The stage, by Ni...",0,0,516,97.02,57.68,...,6.98,0.0,0.97,0.78,0.39,0.39,0,0.39,0.00,0.78
6,https://www.metacritic.com/movie/bronson,8,'Tokyochuchu',"'Nov 30, 2013'","'Bronson is a great, extremely strange movie ...",0,0,94,95.03,70.27,...,6.38,0.0,0.00,0.00,1.06,0.00,0,2.13,0.00,0.00
7,https://www.metacritic.com/movie/bronson,7,'Fenrir81',"'Apr 22, 2011'",'Despite being flawed by a weak narrative str...,0,0,89,98.31,63.20,...,8.99,0.0,0.00,0.00,0.00,0.00,0,2.25,0.00,0.00
8,https://www.metacritic.com/movie/bronson,4,'cabrita',"'Aug 10, 2012'",'If valhalla rising was a weak version of agg...,0,0,115,89.38,21.69,...,0.00,0.0,0.00,0.00,0.00,0.00,0,1.74,0.00,0.00
9,https://www.metacritic.com/movie/bronson,10,'farknash',"'Aug 19, 2011'",'http://mikesharkey.blogspot.com/2009/04/igiz...,0,0,15,97.53,50.00,...,6.67,0.0,0.00,0.00,0.00,6.67,0,13.33,0.00,40.00


In [57]:
#clean the data by removing any rows with missing values
clean_reviews = user_reviews.copy()
clean_reviews.columns = clean_reviews.columns.str.strip().str.lower().str.replace(" ", "_")

clean_reviews["idvscore"] = pd.to_numeric(clean_reviews["idvscore"], errors='coerce')

clean_reviews["reviewer"] = clean_reviews["reviewer"].astype("string").str.strip()
clean_reviews["url"] = clean_reviews["url"].astype("string").str.strip()

display(clean_reviews.isna().sum().sort_values(ascending=False))

thumbsup     3580
thumbstot    3576
datep        3413
rev          3413
idvscore     3409
             ... 
number          0
interrog        0
compare         0
adj             0
otherp          0
Length: 100, dtype: int64

I learned how to clean the data. 
1. Standardize column names and data types.
2. Convert idvscore to numeric values.
3. Inspect missing values by column.
4. Investigate duplicate rows before deleting them.
5. Normalize reviewer and URL values.
6. Add a review_id primary key after deciding how duplicates should be handled.
7. Build the ERD from the cleaned columns.

importance of a surrogate key: to uniquely identify each row. 

giving every review a guranteed unique identifier. and provide a stable primary key even when columns such as 'reviewer', 'url', or review text repeeat. 

In [58]:
clean_reviews.insert(0, "reviewer_id", range(1, len(clean_reviews) + 1))

In [59]:
user_reviews["idvscore_numeric"] = pd.to_numeric(
    user_reviews["idvscore"],
    errors="coerce"
)

top_reviews = (
    user_reviews
    .dropna(subset=["idvscore_numeric"])
    .sort_values("idvscore_numeric", ascending=False)
    [["url", "reviewer", "idvscore_numeric"]]
    .head(10)
)

display(top_reviews)

,url,reviewer,idvscore_numeric
159532,https://www.metacritic.com/movie/your-name,'GuetguetLeBG',10.0
153164,https://www.metacritic.com/movie/gaga-five-foo...,'caiopsico',10.0
153162,https://www.metacritic.com/movie/gaga-five-foo...,'VLee',10.0
153161,https://www.metacritic.com/movie/gaga-five-foo...,'Mrjon',10.0
153160,https://www.metacritic.com/movie/gaga-five-foo...,'Francisco25luis',10.0
153159,https://www.metacritic.com/movie/gaga-five-foo...,'CesarMenezes',10.0
153158,https://www.metacritic.com/movie/gaga-five-foo...,'guizuntini',10.0
153157,https://www.metacritic.com/movie/gaga-five-foo...,'ROMERITO',10.0
153156,https://www.metacritic.com/movie/gaga-five-foo...,'yedu16',10.0
153155,https://www.metacritic.com/movie/gaga-five-foo...,'Itswellison',10.0


In [60]:
df = pd.DataFrame(user_reviews)
df
num_of_rows = len(df)


In [61]:
print(user_reviews.shape)

(319662, 101)


In [62]:
summary = pd.DataFrame({
        'Total Reviews': [len(user_reviews)],
        'Unique Users': [user_reviews.nunique()],
        'Unique URLs':[user_reviews['url'].nunique()],
        'Unique Reviewers':[user_reviews['reviewer'].nunique()],
        'Missing Values': [user_reviews.isnull().sum().sum()],
        'Duplicate Values': [user_reviews.duplicated().sum()],
        "Duplicate Rows": [user_reviews.duplicated().sum()]
})

display(summary)

,Total Reviews,Unique Users,Unique URLs,Unique Reviewers,Missing Values,Duplicate Values,Duplicate Rows
0,319662,url 11364 idvscore ...,11364,106795,24202,922,922


In [63]:
print(user_reviews["idvscore_numeric"].describe())

count    316253.000000
mean          6.608184
std           3.185721
min           0.000000
25%           5.000000
50%           7.000000
75%           9.000000
max          10.000000
Name: idvscore_numeric, dtype: float64


In [64]:
scores = pd.to_numeric(
    user_reviews["idvscore"],
    errors="coerce"
)

invalid_scores = user_reviews.loc[
    scores.isna(),
    "idvscore"
]

print(invalid_scores.value_counts(dropna=False).head(1))

idvscore
NaN    3404
Name: count, dtype: int64


In [65]:
df = pd.read_excel(r'/Users/yg/Documents/Database management/Data Files - database/UserReviewsClean43LIWC.xlsx')

df_raw = df.copy()

df = df.dropna(subset=['reviewer'])
df = df.drop_duplicates()

print("---Cleaned Data Structure ---")
print(df.info())

print("\---Remaining Missing Values---")
print(df.isna().sum())

<>:11: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
<>:11: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
/var/folders/ry/rp7j6v1j37n88zy0_7jq5bjw0000gn/T/ipykernel_61650/2123156920.py:11: SyntaxWarning: "\-" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\-"? A raw string is also an option.
  print("\---Remaining Missing Values---")


---Cleaned Data Structure ---
<class 'pandas.DataFrame'>
Index: 316131 entries, 0 to 319661
Data columns (total 100 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   url           316131 non-null  str    
 1   idvscore      316130 non-null  object 
 2   reviewer      316131 non-null  str    
 3   dateP         316125 non-null  str    
 4   Rev           316125 non-null  str    
 5   thumbsUp      315957 non-null  object 
 6   thumbsTot     315958 non-null  object 
 7   WC            316131 non-null  int64  
 8   Analytic      316131 non-null  float64
 9   Clout         316131 non-null  float64
 10  Authentic     316131 non-null  float64
 11  Tone          316131 non-null  float64
 12  WPS           316131 non-null  float64
 13  Sixltr        316131 non-null  float64
 14  Dic           316131 non-null  float64
 15  function      316131 non-null  float64
 16  pronoun       316131 non-null  float64
 17  ppron         316131 non-null  fl

check for: 
1. missing values
2. duplicates
3. assigning the movie based on that review. movie id = harmonization 
4. year / title / reviews (e.g. Robocop 1980 and Robocop 2014) -> Reviews can be mixed up. 
5. check what transformation that I still need. 
6. Get the structure

In [66]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

string_cols = df.select_dtypes(include=['object', 'string']).columns
for col in string_cols:
    df[col]=(
        df[col].astype(str).str.strip().str.strip("'\""))


#1 Profile the raw Dataset

In [67]:
# Use the dataframe already loaded in the notebook instead of reading a missing raw-file path.
# This avoids the FileNotFoundError and keeps the analysis consistent with the existing dataset.
if "user_reviews" in globals():
    user_raw = user_reviews.copy()
else:
    user_raw = pd.read_excel(
        r"/Users/yg/Documents/Database management/Data Files - database/UserReviewsClean43LIWC.xlsx",
        engine="openpyxl",
    )

#1. dimensions
print(f"shape: {user_raw.shape}")

#2. column types & non-null counts
user_raw.info(verbose=False)

#3. quick check for missing values and compelte row duplicates
print("missing values across dataset:", user_raw.isna().sum().sum())
print("exact duplicate rows:", user_raw.duplicated().sum())

#4. check unique counts for core fields
print("Unique URLs:", user_raw['url'].nunique() if 'url' in user_raw.columns else "N/A")
print("Unique Reviewers:", user_raw['reviewer'].nunique() if 'reviewer' in user_raw.columns else "N/A")

shape: (319662, 101)
<class 'pandas.DataFrame'>
RangeIndex: 319662 entries, 0 to 319661
Columns: 101 entries, url to idvscore_numeric
dtypes: float64(92), int64(2), object(3), str(4)
memory usage: 246.3+ MB
missing values across dataset: 24202
exact duplicate rows: 922
Unique URLs: 11364
Unique Reviewers: 106795


#2 Clean & Rename Columns

In [68]:
user_clean = user_raw.copy()
#Column mapping table
column_mapping = {
    'url': 'url',
    'idvscore': 'review_score', 
    'reviewer': 'reviewer',
    'dateP': 'review_date',
    'Rev': 'review_text',
    'thumbsUp': 'thumbs_up',
    'thumbsTot': 'thumbs_total'
}

user_clean = user_clean.rename(columns=column_mapping)

#standardize LIWC columns (lowercase, spaces to underscores)
liwc_columns = [c for c in user_clean.columns if 'c' not in column_mapping.values()]
user_clean.columns = [c.lower().replace(' ', '_') if c in liwc_columns else c for c in user_clean.columns]

Step 3: Clean Text Fields

In [69]:
text_fields = ['reviewer', 'review_text', 'url']

for col in text_fields:
    if col in user_clean.columns:
        user_clean[col] = (
            user_clean[col]
            .astype("string")
            .str.strip()
            .str.strip("'\"") # Removes wrapping quotes (e.g. 'Longbottom94' -> Longbottom94)[cite: 1]
        .replace(['nan', 'None', 'NULL', ''], np.nan) #ensure missing values stay NaN)
        )

In [70]:
#Step 4: Clean Review Score(review_score)

In [71]:
#conver to float
user_clean['review_score'] = pd.to_numeric(user_clean['review_score'], errors='coerce')

#profile score distribution
print(user_clean['review_score'].describe())

#identify out-of-range values
invalid_scores = user_clean[
    (user_clean['review_score'] < 0) | 
    (user_clean['review_score'] > 10)
]
print(f"Invalid scores out of 0-10 range: {len(invalid_scores)}")

#Flag invalid scores as NaN/NULL
user_clean.loc[
    (user_clean['review_score'] < 0) | 
    (user_clean['review_score'] > 10), 
    'review_score'
] = np.nan

count    316253.000000
mean          6.608184
std           3.185721
min           0.000000
25%           5.000000
50%           7.000000
75%           9.000000
max          10.000000
Name: review_score, dtype: float64
Invalid scores out of 0-10 range: 0


Step 5: Clean Review Dates (review_date )

In [72]:
user_clean['review_date'] = pd.to_datetime(user_clean['review_date'], errors='coerce')

#Verify date range
print(f"Min Date: {user_clean['review_date'].min()}")
print(f"Max Date: {user_clean['review_date'].max()}")
print(f"Unparseable dates (NaT): {user_clean['review_date'].isna().sum()}")

Min Date: 2001-01-26 00:00:00
Max Date: 2022-03-03 00:00:00
Unparseable dates (NaT): 3413


Step 6: Clean Thumbs Up / Thumbs Total

In [73]:
user_clean['thumbs_up'] = pd.to_numeric(user_clean['thumbs_up'], errors='coerce') 
user_clean['thumbs_total'] = pd.to_numeric(user_clean['thumbs_total'], errors='coerce')

#Check logical constraint: thumbs_up <= thumbs_total
invalid_thumbs = user_clean[user_clean['thumbs_up'] > user_clean['thumbs_total']]
print(f"Invalid thumbs_up > thumbs_total: {len(invalid_thumbs)}")

#Nullify invalid thumbs data
user_clean.loc[user_clean['thumbs_up'] > user_clean['thumbs_total'], ['thumbs_up', 'thumbs_total']] = np.nan

Invalid thumbs_up > thumbs_total: 0


Step 7: Check & Convert LIWC Columns

In [74]:
#Extract all LIWC feature metric column names
liwc_columns = [c for c in user_clean.columns if c not in['url', 'review_score', 'reviewer', 'review_date', 'review_text', 'thumbs_up', 'thumbs_total']]

for col in liwc_columns:
    #If read as string with commas (e.g. '12,25') replace comma with dot
    if user_clean[col].dtype == 'object':
        user_clean[col] = user_clean[col].str.replace(',', '.', regex=False)

        #Cast to numeric, keeping unparseable strings as NaN (NULL)
        user_clean[col] = pd.to_numeric(user_clean[col], errors='coerce')

print(f"Processed {len(liwc_columns)} LIWC metric columns.")                                                                                                

Processed 94 LIWC metric columns.


Step #8: Check & handle duplicate reviews

In [75]:
#Check exact duplicates
print("Exact row duplicates:", user_clean. duplicated().sum())

#Check logical duplicates (same reviewer writing a review for the same movie on the same date)
logical_duplicates = user_clean[user_clean.duplicated(subset=['url', 'reviewer', 'review_date', 'review_text'], keep='first')]

print(f"Logical duplicate reviews: {logical_duplicates.count()}")

#Drop true duplicate records
user_clean = user_clean.drop_duplicates(
    subset=['url', 'reviewer', 'review_date', 'review_text']
).reset_index(drop=True)

Exact row duplicates: 926
Logical duplicate reviews: url                 937
review_score        135
reviewer            135
review_date         135
review_text         135
                   ... 
quote               937
apostro             937
parenth             937
otherp              937
idvscore_numeric    135
Length: 101, dtype: int64


Step 9: Add consumer_review_id (PK)

In [76]:
user_clean.insert(0, "reviewer_id", range(1, len(user_clean) + 1))
print(f"Primary Key assigned. Final dataset shape: {user_clean.shape}")

user_clean.head()

Primary Key assigned. Final dataset shape: (318725, 102)


,reviewer_id,url,review_score,reviewer,review_date,review_text,thumbs_up,thumbs_total,wc,analytic,...,colon,semic,qmark,exclam,dash,quote,apostro,parenth,otherp,idvscore_numeric
0,1,https://www.metacritic.com/movie/bronson,8.0,Longbottom94,2013-04-25,Many have dismissed this film for not explorin...,2.0,2.0,113,94.05,...,0.0,0.0,0.0,0.0,0.88,0,1.77,0.00,0.0,8.0
1,2,https://www.metacritic.com/movie/bronson,9.0,MartinB,2009-10-13,Anyone who doesn t like this movie simply just...,0.0,1.0,103,57.11,...,0.0,0.0,0.0,0.0,0.00,0,1.94,5.83,0.0,9.0
2,3,https://www.metacritic.com/movie/bronson,10.0,Jaakko,2012-07-19,Not sure what to think at this film at first. ...,1.0,1.0,143,75.73,...,0.0,0.0,0.0,0.0,0.00,0,1.40,0.00,1.4,10.0
3,4,https://www.metacritic.com/movie/bronson,6.0,CapoR,2009-10-13,Nicely portrayed but it lacks the elements to ...,0.0,1.0,14,83.81,...,0.0,0.0,0.0,0.0,0.00,0,14.29,0.00,0.0,6.0
4,5,https://www.metacritic.com/movie/bronson,8.0,OrwellB.,2009-10-10,Bronson is more than entertainment. It is art ...,0.0,0.0,20,93.26,...,0.0,0.0,0.0,0.0,0.00,0,10.00,0.00,0.0,8.0


In [77]:
movietable = pd.read_csv(r'/Users/yg/Documents/Database management/Movie.csv')

In [78]:
moviecolumn = [('movie_id', 'url')]

In [79]:
moviecolumn = movietable[["url", "movie_id"]]

user_clean = user_clean.merge(
    moviecolumn,
    on="url",
    how="left"
)

In [80]:
user_clean.shape

(318725, 103)

In [81]:
user_clean['movie_id'].isna().sum()

np.int64(0)

In [82]:
user_clean.to_csv ('user_reviews.csv', index = False)

In [1]:
import pandas as pd
import numpy as np

# 1. Load movie master & user reviews raw files
movie_master_path = r'/Users/yg/Documents/Database management/Movie.csv'
user_reviews_path = r'/Users/yg/Documents/Database management/Data Files - database/UserReviewsClean43LIWC.xlsx'

df_movies = pd.read_csv(movie_master_path)
df_raw = pd.read_excel(user_reviews_path)

user_clean = df_raw.copy()

# 2. Clean and standardize column names
column_mapping = {
    'url': 'url',
    'idvscore': 'review_score',
    'reviewer': 'reviewer',
    'dateP': 'review_date',
    'Rev': 'review_text',
    'thumbsUp': 'thumbs_up',
    'thumbsTot': 'thumbs_total'
}

# Rename core columns
user_clean = user_clean.rename(columns=column_mapping)

# Standardize LIWC metric columns to lowercase_snake_case
core_cols = list(column_mapping.values())
user_clean.columns = [
    col if col in core_cols else col.strip().lower().replace(" ", "_") 
    for col in user_clean.columns
]

# 3. Clean text fields (strip whitespace and quotes)
text_fields = ['reviewer', 'review_text', 'url']
for col in text_fields:
    if col in user_clean.columns:
        user_clean[col] = (
            user_clean[col]
            .astype(str)
            .str.strip()
            .str.strip("'\"")
            .replace(['nan', 'None', 'NULL', ''], np.nan)  # FIXED: np.nan
        )

# 4. Clean & validate review score (0-10 scale)
user_clean['review_score'] = pd.to_numeric(user_clean['review_score'], errors='coerce')
# Nullify out-of-range scores
user_clean.loc[(user_clean['review_score'] < 0) | (user_clean['review_score'] > 10), 'review_score'] = np.nan

# 5. Convert Dates to ISO format (YYYY-MM-DD)
user_clean['review_date'] = pd.to_datetime(user_clean['review_date'], errors='coerce')

# 6. Clean thumbs up / thumbs total
user_clean['thumbs_up'] = pd.to_numeric(user_clean['thumbs_up'], errors='coerce')
user_clean['thumbs_total'] = pd.to_numeric(user_clean['thumbs_total'], errors='coerce')

# Enforce logical constraint: thumbs_up <= thumbs_total
invalid_thumbs = user_clean['thumbs_up'] > user_clean['thumbs_total']
user_clean.loc[invalid_thumbs, ['thumbs_up', 'thumbs_total']] = np.nan

# 7. Process LIWC columns
liwc_columns = [c for c in user_clean.columns if c not in core_cols]

for col in liwc_columns:
    if user_clean[col].dtype == 'object':
        user_clean[col] = user_clean[col].astype(str).str.replace(',', '.', regex=False)
    user_clean[col] = pd.to_numeric(user_clean[col], errors='coerce')

# 8. Deduplicate Logical Reviews
user_clean = user_clean.drop_duplicates(
    subset=['url', 'reviewer', 'review_date', 'review_text']
).reset_index(drop=True)

# 9. Map 'movie_id' from Movie.csv using URL
# Standardize URL in both dataframes for clean joining
df_movies['url'] = df_movies['url'].astype(str).str.strip().str.strip("'\"").str.lower()  # FIXED: added .str.lower()
user_clean['url'] = user_clean['url'].astype(str).str.strip().str.strip("'\"").str.lower()  # FIXED: added missing dot before str.lower()

# Left merge movie_id
user_clean = user_clean.merge(
    df_movies[['movie_id', 'url']], 
    on='url', 
    how='left'
)

# Check match stats
matched_count = user_clean['movie_id'].notna().sum()
print(f"Successfully matched {matched_count} out of {len(user_clean)} reviews to movie_id.")

# 10. Assign primary key & export CSV
# Drop URL (no longer needed in Consumer_Reviews table since movie_id is FK)
if 'url' in user_clean.columns:
    user_clean = user_clean.drop(columns=['url'])

# Assign auto-incrementing primary key as column 0
user_clean.insert(0, "consumer_review_id", range(1, len(user_clean) + 1))

# Reorder: PK, FK, Core Fields, LIWC Fields
ordered_columns = ['consumer_review_id', 'movie_id', 'reviewer', 'review_date', 'review_score', 'review_text', 'thumbs_up', 'thumbs_total'] + liwc_columns  # FIXED: liwc_columns
consumer_reviews_df = user_clean[ordered_columns]

# Export to CSV
consumer_reviews_df.to_csv('Consumer_Reviews.csv', index=False)
print("Successfully generated 'Consumer_Reviews.csv'!")

Successfully matched 318725 out of 318725 reviews to movie_id.
Successfully generated 'Consumer_Reviews.csv'!
